# 세부목적 4 · ASOS 강수 규모별 GPM–ASOS 편차
**측량학(2) 1조 | 2020~2024년 여름철(6~8월), 부산(159)·울산(152)·창원(155)**

**목적**: ASOS 강수 규모에 따라 GPM-IMERG 일강수량 오차의 **방향(Bias)**과 **크기(MAE, RMSE)**가 각 지점에서 어떻게 달라지는지 분석한다.

**입력**: 공통 통합자료 `MASTER_GPM_ASOS_2020_2024_JJA_1380_corrected.csv` (1,380건).

**구간 정의**: ① ASOS=0mm, ② 0<ASOS<10mm, ③ 10≤ASOS<30mm, ④ ASOS≥30mm. **10.0mm는 3구간, 30.0mm는 4구간**.

**지표**: $d_i=GPM_i-ASOS_i$; Bias=$\overline{d}$; MAE=$\overline{|d|}$; RMSE=$\sqrt{\overline{d^2}}$.

기존 통합자료의 ASOS 공란을 0mm로 채운 전처리 결과를 사용한다. 원본 기상관측의 공란을 본 노트북에서 재해석하지 않는다. 내장 CSV로 Colab에서 추가 파일 없이 실행 가능.


In [ ]:
import sys, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from pathlib import Path
from IPython.display import display
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)
STATIONS = ['부산','울산','창원']
NAME_EN = {'부산':'Busan','울산':'Ulsan','창원':'Changwon'}
print(f'Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}')


In [ ]:
# 보고서에 사용한 원본/중간 산출 CSV를 gzip으로 내장한 스냅샷 (2026-11-26 자료)
# GitHub의 data/processed/에 같은 파일이 있으면 그것을 우선 사용합니다.
import base64, gzip, io
from pathlib import Path
EMBEDDED_DATA = {
    'MASTER_GPM_ASOS_2020_2024_JJA_1380_corrected.csv': (
        '...'
    ),
}
def load_study_csv(filename):
    paths = [Path('data/processed') / filename, Path('../data/processed') / filename, Path(filename)]
    for p in paths:
        if p.is_file():
            print(f'입력 파일: {p}')
            return pd.read_csv(p, encoding='utf-8-sig')
    payload = ''.join(EMBEDDED_DATA[filename])
    raw = gzip.decompress(base64.b64decode(payload))
    print(f'내장된 연구용 CSV 사용: {filename}')
    return pd.read_csv(io.BytesIO(raw), encoding='utf-8-sig')


In [ ]:
master = load_study_csv('MASTER_GPM_ASOS_2020_2024_JJA_1380_corrected.csv')
required={'stn','name_kr','date_kst','asos_mm_filled','gpm_cal_mm','half_hour_count'}
assert required.issubset(master.columns),f'입력 CSV 열 부족: {required-set(master.columns)}'
assert len(master)==1380 and all(master.groupby('name_kr').size().reindex(STATIONS)==460)
assert (master['half_hour_count']==48).all()
assert master['asos_mm_filled'].notna().all() and master['gpm_cal_mm'].notna().all()
assert (master['asos_mm_filled']>=0).all()
print('대상',len(master),'건, 날짜',master['date_kst'].nunique(),'일, 지점',master['name_kr'].unique())


## 1. 구간을 원자료 값에서 다시 분류
기존 CSV의 `rain_category` 텍스트를 그대로 믿지 않고 ASOS 보정값에서 직접 분류한다. 경계값(10.0mm, 30.0mm)에 오류가 없는지 점검한다.


In [ ]:
df=master.copy()
df['ASOS_mm']=pd.to_numeric(df['asos_mm_filled'])
df['GPM_mm']=pd.to_numeric(df['gpm_cal_mm'])
df['diff_mm']=df['GPM_mm']-df['ASOS_mm']
df['abs_err_mm']=df['diff_mm'].abs()
df['sq_err_mm']=df['diff_mm']**2
LABELS=['0 mm','0–<10 mm','10–<30 mm','>=30 mm']
bins=[-1e-12,0,10,30,np.inf]
df['class']=pd.cut(df['ASOS_mm'],bins=bins,labels=LABELS,include_lowest=True,right=False)
# 0mm 별도 보정: ASOS=0만 무강수일, 0초과~10미만은 약한 강수
df.loc[df['ASOS_mm']==0,'class']='0 mm'
assert df['class'].notna().all()
assert (df.loc[df['ASOS_mm']==10,'class']=='10–<30 mm').all()
assert (df.loc[df['ASOS_mm']==30,'class']=='>=30 mm').all()
counts=df['class'].value_counts().reindex(LABELS)
print('구간별 표본수');display(counts.rename('n').to_frame())
assert counts.tolist()==[828,289,124,139],f'구간별 표본수 불일치: {counts.tolist()}'
print('기존 연구의 총 1,380건 및 경계값 분류와 일치')


## 2. 지점·강수구간별 Bias / MAE / RMSE 재산정
**기여도 정의**: 강수구간의 $d_i$ 합계를 **각 지점 전체 460일**로 나눈 값(mm/일)이다. 구간별 평균 Bias와는 다르다. 구간별 기여도의 합은 해당 지점의 전체 Bias와 같아야 한다.


In [ ]:
def summarize(g):
    d=g['diff_mm'].to_numpy(dtype=float)
    return pd.Series({'n':len(g),'ASOS_mean_mm':g['ASOS_mm'].mean(),
      'GPM_mean_mm':g['GPM_mm'].mean(), 'Bias_mm':d.mean(),
      'MAE_mm':np.abs(d).mean(),'RMSE_mm':np.sqrt(np.mean(d*d)),
      'Over_days':int((d>1e-9).sum()),'Equal_days':int((np.abs(d)<=1e-9).sum()),
      'Under_days':int((d< -1e-9).sum()),'Error_sum_mm':d.sum()})
rows=[]
for site in STATIONS:
    for category in LABELS:
        part=df[(df['name_kr']==site)&(df['class']==category)]
        stats=summarize(part)
        rows.append({'지점':site,'강수구간':category,**stats.to_dict(),
                     'Bias_contribution_mm_per_day':stats['Error_sum_mm']/460})
by_bin=pd.DataFrame(rows)
by_site=pd.DataFrame([{'지점':site,**summarize(df[df['name_kr']==site]).to_dict()} for site in STATIONS])
print('지점별 전체 통계');display(by_site.round(3))
print('지점×강수구간 통계');display(by_bin.round(3))
for site in STATIONS:
    b=by_bin[by_bin['지점']==site]
    whole=by_site.set_index('지점').loc[site,'Bias_mm']
    assert np.isclose(b['Bias_contribution_mm_per_day'].sum(),whole,atol=1e-10)
assert np.allclose(by_site['Bias_mm'],[-2.055076,.711380,-.701109],atol=.001)
assert np.allclose(by_bin[by_bin['강수구간']=='>=30 mm']['Bias_mm'],[-28.826636,-7.254461,-13.772801],atol=.001)
by_bin.to_csv(OUT/'04_rain_bin_bias_mae_rmse.csv',index=False,encoding='utf-8-sig')
by_site.to_csv(OUT/'04_station_summary.csv',index=False,encoding='utf-8-sig')


## 3. 전체 1,380건 통계 및 강수 구간별 빈도
4개 구간에서 각각 얼마나 많은 사례가 있는지, 전체 Bias가 왜 작게 보일 수 있는지 확인한다.


In [ ]:
entire=summarize(df)
r=pearsonr(df['GPM_mm'],df['ASOS_mm']).statistic
print(f'전체 n={len(df)}, r={r:.3f}, Bias={entire["Bias_mm"]:.2f}mm, MAE={entire["MAE_mm"]:.2f}mm, RMSE={entire["RMSE_mm"]:.2f}mm')
assert np.isclose(r,0.787,atol=.002)
assert np.isclose(entire['Bias_mm'],-.68160,atol=.001)
assert np.isclose(entire['MAE_mm'],5.56226,atol=.001)
assert np.isclose(entire['RMSE_mm'],15.00309,atol=.002)
print('전체 결과도 보고서와 일치')


## 4. 그림 저장
GitHub 보고서의 그림에 해당하는 네 종류의 도표를 같은 분석 기준으로 다시 만든다. 그림 디자인은 보고서에 이미 업로드한 최종 도표와 동일하지 않을 수 있으나 **통계 입력값은 동일**하다.


In [ ]:
def grouped_plot(metric,ylabel,title,filename):
    piv=by_bin.pivot(index='강수구간',columns='지점',values=metric).reindex(LABELS)[STATIONS]
    ax=piv.rename(columns=NAME_EN).plot(kind='bar',figsize=(9,5),rot=0)
    ax.set(xlabel='ASOS daily rainfall class',ylabel=ylabel,title=title)
    ax.grid(axis='y',alpha=.2)
    ax.legend(title='Station')
    if metric in ('Bias_mm','Bias_contribution_mm_per_day'): ax.axhline(0,color='black',lw=.8)
    plt.tight_layout();plt.savefig(OUT/filename,dpi=180);plt.show()
grouped_plot('Bias_mm','Bias (mm)','Subgoal 4 | Mean bias by rainfall class','04_bias_by_rain.png')
# MAE & RMSE는 보고서 방식대로 동일 축에 두되 구간별 별도 그림도 저장 가능
for metric,fn in [('MAE_mm','04_mae_by_rain.png'),('RMSE_mm','04_rmse_by_rain.png')]:
    grouped_plot(metric,metric.replace('_',' '),f'Subgoal 4 | {metric} by rainfall class',fn)
grouped_plot('Bias_contribution_mm_per_day','Contribution to 460-day Bias (mm/day)',
             'Subgoal 4 | Contribution of each rainfall class to station Bias','04_bias_contribution.png')
grouped_plot('n','Number of station-days','Subgoal 4 | Sample size by rainfall class','04_samples.png')


In [ ]:
# 분석 결과 파일 목록
print('\n'.join(str(f) for f in sorted(OUT.glob('04_*'))))
print('보고서 대조: 0<ASOS<10mm 3지점 양의 Bias / ASOS≥30mm 3지점 음의 Bias')


## 5. 해석상의 범위
- `GPM_mm`은 이미 UTC→KST 날짜 일치·30분 자료 적산을 거친 **기존 통합 마스터의 값**이다. 이 노트북에서 IMERG를 새로 받거나 적산하지 않는다.
- ASOS 원자료 공란 689건을 0mm로 보정한 기존 연구의 가정을 따른다. '무강수' 분류와 통계도 이 가정에 영향을 받는다.
- 10.0mm와 30.0mm 경계는 중복 없이 자동 분류한다.
- `Bias_contribution_mm_per_day`는 **지점별 460일**을 분모로 계산하므로 구간별 Bias 값과 구분한다.
- 통계표와 그림은 `outputs/`에 CSV/PNG로 저장한다.
